In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from scipy import sparse
import pandas as pd
import gc

In [4]:
train = pd.read_csv('./elo-merchant-category-recommendation/xgboost/train.csv')
test = pd.read_csv('./elo-merchant-category-recommendation/xgboost/test.csv')
merchant = pd.read_csv('./elo-merchant-category-recommendation/merchants.csv')
new_transaction = pd.read_csv('./elo-merchant-category-recommendation/new_merchant_transactions.csv')
history_transactions = pd.read_csv('./elo-merchant-category-recommendation/historical_transactions.csv')
transaction = pd.concat([new_transaction, history_transactions], axis=0, ignore_index=True)
del new_transaction, history_transactions
gc.collect()

0

In [5]:
nlp_features = ['merchant_id', 'merchant_category_id', 'state_id', 'subsector_id', 'city_id']

for co in nlp_features:
    print(co)
    transaction[co] = transaction[co].astype(str)
    temp = transaction[transaction['month_lag']>=0].groupby("card_id")[co].apply(list).apply(lambda x:' '.join(x)).reset_index()
    temp.columns = ['card_id', co+'_new']
    train = pd.merge(train, temp, how='left', on='card_id')
    test = pd.merge(test, temp, how='left', on='card_id')

    temp = transaction[transaction['month_lag']<0].groupby("card_id")[co].apply(list).apply(lambda x:' '.join(x)).reset_index()
    temp.columns = ['card_id', co+'_hist']
    train = pd.merge(train, temp, how='left', on='card_id')
    test = pd.merge(test, temp, how='left', on='card_id')

    temp = transaction.groupby("card_id")[co].apply(list).apply(lambda x:' '.join(x)).reset_index()
    temp.columns = ['card_id', co+'_all']
    train = pd.merge(train, temp, how='left', on='card_id').fillna("-1")
    test = pd.merge(test, temp, how='left', on='card_id').fillna("-1")

merchant_id
merchant_category_id
state_id
subsector_id
city_id


In [9]:
# Initialize sparse matrix
train_x = pd.DataFrame()
test_x = pd.DataFrame()
# Vectorizer definition
cntv = CountVectorizer()
tfv = TfidfVectorizer(
    ngram_range=(1, 2), 
    min_df=3, 
    max_df=0.9, 
    use_idf=True, 
    smooth_idf=True, 
    sublinear_tf=True
)

# Build NLP feature name list
base_features = ['merchant_id', 'merchant_category_id', 'state_id', 'subsector_id', 'city_id']
vector_features = [f"{feat}_{suffix}" for feat in base_features for suffix in ['new', 'hist', 'all']]

# Start processing each text feature
for feature in vector_features:
    print(f"Processing feature: {feature}")

    # Fill missing values
    train[feature] = train[feature].fillna("")
    test[feature] = test[feature].fillna("")

    # === CountVectorizer ===
    cntv.fit(train[feature].tolist() + test[feature].tolist())
    train_vec = cntv.transform(train[feature])
    test_vec = cntv.transform(test[feature])
    
    train_x = sparse.hstack([train_x, train_vec]).tocsr()
    test_x = sparse.hstack([test_x, test_vec]).tocsr()

    # === TF-IDF Vectorizer ===
    tfv.fit(train[feature].tolist() + test[feature].tolist())
    train_vec = tfv.transform(train[feature])
    test_vec = tfv.transform(test[feature])

    train_x = sparse.hstack([train_x, train_vec]).tocsr()
    test_x = sparse.hstack([test_x, test_vec]).tocsr()
    
sparse.save_npz("./elo-merchant-category-recommendation/xgboost/train_nlp.npz", train_x)
sparse.save_npz("./elo-merchant-category-recommendation/xgboost/test_nlp.npz", test_x)

Processing feature: merchant_id_new
Processing feature: merchant_id_hist
Processing feature: merchant_id_all
Processing feature: merchant_category_id_new
Processing feature: merchant_category_id_hist
Processing feature: merchant_category_id_all
Processing feature: state_id_new
Processing feature: state_id_hist
Processing feature: state_id_all
Processing feature: subsector_id_new
Processing feature: subsector_id_hist
Processing feature: subsector_id_all
Processing feature: city_id_new
Processing feature: city_id_hist
Processing feature: city_id_all


In [10]:
train_x.shape

(201917, 4235134)

In [13]:
import xgboost as xgb
from sklearn.feature_selection import f_regression
from numpy.random import RandomState
from bayes_opt import BayesianOptimization

In [14]:
train = pd.read_csv('./elo-merchant-category-recommendation/train.csv')
test = pd.read_csv('./elo-merchant-category-recommendation/test.csv')

In [ ]:
features = train.columns.tolist()
features.remove('card_id')
features.remove('target')

train_x = sparse.load_npz("./elo-merchant-category-recommendation/xgboost/train_nlp.npz")
test_x = sparse.load_npz("./elo-merchant-category-recommendation/xgboost/test_nlp.npz")

train_x = sparse.hstack((train_x, train[features])).tocsr()
test_x = sparse.hstack((test_x, test[features])).tocsr()

In [ ]:
def params_append(params):
    """

    :param params:
    :return:
    """
    params['objective'] = 'reg:squarederror'
    params['eval_metric'] = 'rmse'
    params["min_child_weight"] = int(params["min_child_weight"])
    params['max_depth'] = int(params['max_depth'])
    return params

# Model optimization function
def param_beyesian(train):
    """

    :param train:
    :return:
    """
    # Part 1. Data preparation
    train_y = pd.read_csv("data/train.csv")['target']
    # Data encapsulation
    sample_index = train_y.sample(frac=0.1, random_state=2020).index.tolist()
    train_data = xgb.DMatrix(train.tocsr()[sample_index, :
                             ], train_y.loc[sample_index].values, silent=True)
    
    # Build objective function through cross-validation process
    def xgb_cv(colsample_bytree, subsample, min_child_weight, max_depth,
               reg_alpha, eta,
               reg_lambda):
        """

        :param colsample_bytree:
        :param subsample:
        :param min_child_weight:
        :param max_depth:
        :param reg_alpha:
        :param eta:
        :param reg_lambda:
        :return:
        """
        params = {'objective': 'reg:squarederror',
                  'early_stopping_round': 50,
                  'eval_metric': 'rmse'}
        params['colsample_bytree'] = max(min(colsample_bytree, 1), 0)
        params['subsample'] = max(min(subsample, 1), 0)
        params["min_child_weight"] = int(min_child_weight)
        params['max_depth'] = int(max_depth)
        params['eta'] = float(eta)
        params['reg_alpha'] = max(reg_alpha, 0)
        params['reg_lambda'] = max(reg_lambda, 0)
        print(params)
        cv_result = xgb.cv(params, train_data,
                           num_boost_round=1000,
                           nfold=2, seed=2,
                           stratified=False,
                           shuffle=True,
                           early_stopping_rounds=30,
                           verbose_eval=False)
        return -min(cv_result['test-rmse-mean'])
    
    # Call Bayesian optimizer for model optimization
    xgb_bo = BayesianOptimization(
        xgb_cv,
        {'colsample_bytree': (0.5, 1),
         'subsample': (0.5, 1),
         'min_child_weight': (1, 30),
         'max_depth': (5, 12),
         'reg_alpha': (0, 5),
         'eta':(0.02, 0.2),
         'reg_lambda': (0, 5)}
    )
    xgb_bo.maximize(init_points=21, n_iter=5)  # init_points represents initial points, n_iter represents number of iterations (sampling count)
    print(xgb_bo.max['target'], xgb_bo.max['params'])
    return xgb_bo.max['params']

# Cross-validation prediction function
def train_predict(train, test, params):
    """

    :param train:
    :param test:
    :param params:
    :return:
    """
    train_y = pd.read_csv("data/train.csv")['target']
    test_data = xgb.DMatrix(test)

    params = params_append(params)
    kf = KFold(n_splits=5, random_state=2020, shuffle=True)
    prediction_test = 0
    cv_score = []
    prediction_train = pd.Series()
    ESR = 30
    NBR = 10000
    VBE = 50
    for train_part_index, eval_index in kf.split(train, train_y):
        # Model training
        train_part = xgb.DMatrix(train.tocsr()[train_part_index, :],
                                 train_y.loc[train_part_index])
        eval = xgb.DMatrix(train.tocsr()[eval_index, :],
                           train_y.loc[eval_index])
        bst = xgb.train(params, train_part, NBR, [(train_part, 'train'),
                                                          (eval, 'eval')], verbose_eval=VBE,
                        maximize=False, early_stopping_rounds=ESR, )
        prediction_test += bst.predict(test_data)
        eval_pre = bst.predict(eval)
        prediction_train = prediction_train.append(pd.Series(eval_pre, index=eval_index))
        score = np.sqrt(mean_squared_error(train_y.loc[eval_index].values, eval_pre))
        cv_score.append(score)
    print(cv_score, sum(cv_score) / 5)
    pd.Series(prediction_train.sort_index().values).to_csv("preprocess/train_xgboost.csv", index=False)
    pd.Series(prediction_test / 5).to_csv("preprocess/test_xgboost.csv", index=False)
    test = pd.read_csv('data/test.csv')
    test['target'] = prediction_test / 5
    test[['card_id', 'target']].to_csv("result/submission_xgboost.csv", index=False)
    return